## 2_oversampling_examples.ipynb
### Fairness and Compliance of AI
### University of Oklahoma

This script illustrates oversampling techniques to reduce the imbalance of a dataset with respect to fairness attributes.  The dataset used is the UCI Adult Census dataset. We will focus on imbalance associated with the variable 'sex'.

The first code block reads in the dataset and provides some simple exploration.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE, RandomOverSampler
from collections import Counter
import os, random

# Global seeds for reproducibility
os.environ["PYTHONHASHSEED"] = "42"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
random.seed(42)
np.random.seed(42)

# ---------------------------------------------------------
# 1. LOAD ADULT DATASET (COMMON FAIRNESS DATASET)
# ---------------------------------------------------------
df = pd.read_csv("adult_imputed.csv")

# Examine the dataset
print("Dataset head:")
print(df.head())

# Check distribution of records by protected attribute
def dist_tbl(df, groupbylist):
    # 1. Base counts
    tbl = (
        df.groupby(groupbylist)
        .size()
        .reset_index(name="count")
    )

    # 2. Percent of income within sex (row %)
    tbl["pct_within_colA"] = (
        tbl["count"] /
        tbl.groupby(groupbylist[0])["count"].transform("sum")
    )

    # 3. Percent of income × sex out of total (overall %)
    tbl["pct_of_total"] = tbl["count"] / tbl["count"].sum()
    
    return tbl

tbl = dist_tbl(df, ["sex", "target"])
tbl



We see that the dataset is imbalanced with respect to sex -- there are less high income records as a percent of females than of males.  We need to rebalance so P(Y=high | female) = P(Y=high | male).

### Rebalancing with SMOTE (Group Aware-SMOTE) for classification

In [ ]:
# Define the protected attribute (A) and label (Y)
A = df["sex"]                     # protected attribute
Y = df["target"]                  # classification label

# Modeling typically requires numeric encoding, so we encode A and Y as integers.
A_le = LabelEncoder()
Y_le = LabelEncoder()

A_enc = A_le.fit_transform(A)
Y_enc = Y_le.fit_transform(Y)

# The features (X) are all columns except the target and protected attribute.
X = df.drop(columns=["target", "sex"])

# Encode categorical features into numeric form for SMOTE/oversampling.
# In this case, we one-hot encode the categorical features (e.g., "workclass", "education")
X_enc = pd.get_dummies(X, drop_first=True)

In [ ]:
# We can view the encoded features and labels
print(A_enc[:5])
print(Y_enc[:5])
print(X_enc.head())

In [ ]:
# ---------------------------------------------------------
# 2. DEFINE GROUP AWARE-SMOTE FOR CLASSIFICATION
# ---------------------------------------------------------
# Group Aware-SMOTE = SMOTE applied *within each (A,Y) subgroup*

def group_aware_smote(X, A, Y, k_neighbors=5, random_state=42):
    """Perform Group Aware-SMOTE by applying SMOTE separately within each protected group.

    This keeps synthetic samples within the same protected attribute value (A) while
    balancing the target label (Y) inside each group.

    parameters:
    - X: feature matrix (DataFrame)
    - A: protected attribute array (1D)
    - Y: target label array (1D)
    - k_neighbors: number of nearest neighbors for SMOTE
    """
    X_res_list = []
    A_res_list = []
    Y_res_list = []

    # Identify unique protected groups
    groups = sorted(set(A))
    columns = X.columns

    for a_val in groups:
        mask = (A == a_val)
        X_sub = X[mask]
        y_sub = Y[mask]

        # If group does not contain both classes, skip SMOTE (nothing to balance)
        if len(set(y_sub)) < 2 or len(X_sub) < k_neighbors + 1:
            X_res_list.append(X_sub)
            A_res_list.extend([a_val] * len(X_sub))
            Y_res_list.extend(y_sub.tolist())
            continue

        sm = SMOTE(k_neighbors=k_neighbors, random_state=random_state)
        X_res, y_res = sm.fit_resample(X_sub, y_sub)

        X_res_list.append(pd.DataFrame(X_res, columns=columns))
        A_res_list.extend([a_val] * len(X_res))
        Y_res_list.extend(y_res.tolist())

    X_final = pd.concat(X_res_list, ignore_index=True)
    A_final = np.array(A_res_list)
    Y_final = np.array(Y_res_list)

    return X_final, A_final, Y_final


In [ ]:
# ---------------------------------------------------------
# 3. RUN SMOTE AND CHECK RESULTS
# ---------------------------------------------------------
# Group Aware-SMOTE = SMOTE applied *within each (A,Y) subgroup*
X_fair, A_fair, Y_fair = group_aware_smote(X_enc, A_enc, Y_enc, k_neighbors=5, random_state=42)

print("\nAfter Group Aware-SMOTE subgroup counts:")
print(Counter(list(zip(A_fair, Y_fair))))


In [ ]:
# ---------------------------------------------------------
# 4. DECODE GROUP AWARE-SMOTE RESULTS BACK TO ORIGINAL SCHEMA
# ---------------------------------------------------------
# Group Aware-SMOTE = SMOTE applied *within each (A,Y) subgroup*

# 1. Decode protected attribute + label using the SAME encoders
sex_decoded = A_le.inverse_transform(A_fair)
income_decoded = Y_le.inverse_transform(Y_fair)

# 2. Start with the one-hot encoded features
df_fair = pd.DataFrame(X_fair, columns=X_enc.columns)

# 3. Reconstruct original categorical columns from one-hot encoding
#    (this reverses get_dummies)
df_cats = pd.get_dummies(df.drop(columns=["sex", "target"]), drop_first=True)
missing_cols = set(df_cats.columns) - set(df_fair.columns)

# Add any missing dummy columns with zeros (SMOTE sometimes drops rare dummies)
for col in missing_cols:
    df_fair[col] = 0

# Now ensure column order matches the original dummy matrix
df_fair = df_fair[df_cats.columns]

# 4. Add back original categorical columns by decoding from dummies
df_original = df.drop(columns=["sex", "target"])
df_fair_original = pd.DataFrame(index=df_fair.index)

for col in df_original.columns:
    if df_original[col].dtype == "object":
        # extract all dummy columns for this categorical variable
        prefix = col + "_"
        dummies = [c for c in df_fair.columns if c.startswith(prefix)]
        # argmax to recover category
        df_fair_original[col] = df_fair[dummies].idxmax(axis=1).str.replace(prefix, "")
    else:
        # numeric columns pass through unchanged
        df_fair_original[col] = df_fair[col]

# 5. Add decoded sex + income
df_fair_original["sex"] = sex_decoded
df_fair_original["target"] = income_decoded

# 6. Reorder to match original schema
df_fair_original = df_fair_original[df.columns]



In [ ]:
# Check the distribution of records by protected attribute and label after Fair-SMOTE
print(df_fair_original.head())
dist_tbl(df_fair_original, ["sex", "target"])

In [ ]:
# Save the new balanced dataset to a CSV file
df_fair_original.to_csv("adult_fair_smote.csv", index=False)